In [1]:
import pandas as pd
import numpy as np
import itertools
import datetime
import pandas_gbq
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
from pathlib import Path
from PIL import Image
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT
*
FROM `perceptive-ivy-290216.f1_api.sprint_lap_time`  A
# WHERE A.Year=2024
# AND A.GP="Monaco Grand Prix"
# AND A.DRIVER='HAM'
ORDER BY LapNumber, LapStartTime
"""
track3=pandas_gbq.read_gbq(query2,project_id,dialect='standard')

Downloading: 100%|██████████|


In [5]:
track2=track3[(track3["GP"]=='Qatar Grand Prix')&(track3["Year"]==2025)]
track2.head()
year=track2['Year'].iloc[0]
gp=track2['GP'].iloc[0]

In [6]:
track2['LapTime']= pd.to_timedelta(track2["LapTime"])

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_38367/4173085553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2['LapTime']= pd.to_timedelta(track2["LapTime"])


In [7]:
# track2['LapTime'].dt.total_seconds().min()*1.07
track2=track2[track2['LapNumber']!=1.0]

In [8]:
# quicklaps=track2[track2["LapTime"]<track2['LapTime'].min()*1.1]
quicklaps=track2
quicklaps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP
660,0 days 00:48:51.615000,PIA,81,0 days 00:01:25.086000,2.0,1.0,NaT,NaT,0 days 00:00:31.501000,0 days 00:00:28.571000,0 days 00:00:25.014000,0 days 00:47:58.065000,0 days 00:48:26.636000,0 days 00:48:51.650000,242.0,288.0,278.0,300.0,True,MEDIUM,7.0,False,McLaren,0 days 00:47:26.529000,2025-11-29 14:05:00.354,1,1.0,False,,False,True,2025,Qatar Grand Prix
663,0 days 00:48:52.888000,RUS,63,0 days 00:01:25.085000,2.0,1.0,NaT,NaT,0 days 00:00:31.518000,0 days 00:00:28.766000,0 days 00:00:24.801000,0 days 00:47:59.341000,0 days 00:48:28.107000,0 days 00:48:52.908000,NaN,289.0,280.0,293.0,True,MEDIUM,8.0,False,Mercedes,0 days 00:47:27.803000,2025-11-29 14:05:01.628,1,2.0,False,,False,True,2025,Qatar Grand Prix
667,0 days 00:48:54.111000,NOR,4,0 days 00:01:25.135000,2.0,2.0,NaT,NaT,0 days 00:00:31.619000,0 days 00:00:28.844000,0 days 00:00:24.672000,0 days 00:48:00.622000,0 days 00:48:29.466000,0 days 00:48:54.138000,241.0,288.0,280.0,305.0,True,HARD,2.0,True,McLaren,0 days 00:47:28.976000,2025-11-29 14:05:02.801,1,3.0,False,,False,True,2025,Qatar Grand Prix
670,0 days 00:48:54.729000,VER,1,0 days 00:01:25.136000,2.0,2.0,NaT,NaT,0 days 00:00:31.491000,0 days 00:00:29.106000,0 days 00:00:24.539000,0 days 00:48:01.109000,0 days 00:48:30.215000,0 days 00:48:54.754000,244.0,287.0,283.0,302.0,True,HARD,2.0,True,Red Bull Racing,0 days 00:47:29.593000,2025-11-29 14:05:03.418,1,4.0,False,,False,True,2025,Qatar Grand Prix
673,0 days 00:48:56.125000,TSU,22,0 days 00:01:25.493000,2.0,2.0,NaT,NaT,0 days 00:00:31.647000,0 days 00:00:28.978000,0 days 00:00:24.868000,0 days 00:48:02.312000,0 days 00:48:31.290000,0 days 00:48:56.158000,241.0,286.0,281.0,298.0,True,HARD,2.0,True,Red Bull Racing,0 days 00:47:30.632000,2025-11-29 14:05:04.457,1,5.0,False,,False,True,2025,Qatar Grand Prix


In [9]:
#Remove Pitstops and Track Status other than Clear to remove slow laps
quicklaps=quicklaps[((quicklaps["PitOutTime"]=='NaT')&(quicklaps["PitInTime"]=='NaT')&(~quicklaps["TrackStatus"].isin(['4','41','5','6','7','124'])))]

In [10]:
transformed_laps_driver = quicklaps.copy()
transformed_laps_driver.loc[:, "LapTime (s)"] = quicklaps["LapTime"].dt.total_seconds()

# order the team from the fastest (lowest median lap time) tp slower
team_order = (
    transformed_laps_driver[["Driver", "LapTime (s)"]].groupby("Driver").median()["LapTime (s)"].sort_values().index
)
print(team_order)

Index(['PIA', 'RUS', 'NOR', 'VER', 'TSU', 'ANT', 'ALO', 'HAD', 'SAI', 'ALB',
       'LEC', 'BOR', 'LAW', 'BEA', 'OCO', 'HUL', 'STR', 'GAS', 'HAM', 'COL'],
      dtype='object', name='Driver')


In [11]:
transformed_laps_driver['Median (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform('median')
transformed_laps_driver['Median LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform('median')

transformed_laps_driver['Fastest (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform('min')
transformed_laps_driver['Fastest LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform('min')

transformed_laps_driver['Average (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform(np.mean)
transformed_laps_driver['Average LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform(np.mean)

transformed_laps_driver=transformed_laps_driver.fillna(0)

transformed_laps_driver.tail()

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_38367/3103475934.py:7: FutureWarning: The provided callable <function mean at 0x1063eb560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  transformed_laps_driver['Average (s)'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime (s)"].transform(np.mean)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_38367/3103475934.py:8: FutureWarning: The provided callable <function mean at 0x1063eb560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  transformed_laps_driver['Average LapTime'] = transformed_laps_driver.groupby(['Driver','Compound'])["LapTime"].transform(np.mean)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,LapTime (s),Median (s),Median LapTime,Fastest (s),Fastest LapTime,Average (s),Average LapTime
8214,0 days 01:13:31.676000,HUL,27,0 days 00:01:26.522000,19.0,1.0,NaT,NaT,0 days 00:00:31.982000,0 days 00:00:29.564000,0 days 00:00:24.976000,0 days 01:12:37.161000,0 days 01:13:06.725000,0 days 01:13:31.701000,238.0,288.0,278.0,333.0,False,MEDIUM,25.0,False,Kick Sauber,0 days 01:12:05.154000,2025-11-29 14:29:38.979,1,16.0,False,,False,True,2025,Qatar Grand Prix,86.522,86.3615,0 days 00:01:26.361500,85.678,0 days 00:01:25.678000,86.409444,0 days 00:01:26.409444444
8215,0 days 01:13:38.209000,HAM,44,0 days 00:01:28.912000,19.0,1.0,NaT,NaT,0 days 00:00:32.788000,0 days 00:00:29.955000,0 days 00:00:26.169000,0 days 01:12:42.113000,0 days 01:13:12.068000,0 days 01:13:38.237000,239.0,287.0,242.0,309.0,False,MEDIUM,19.0,True,Ferrari,0 days 01:12:09.297000,2025-11-29 14:29:43.122,1,17.0,False,,False,True,2025,Qatar Grand Prix,88.912,86.5675,0 days 00:01:26.567500,85.782,0 days 00:01:25.782000,86.732111,0 days 00:01:26.732111111
8216,0 days 01:14:01.572000,GAS,10,0 days 00:01:23.188000,19.0,2.0,NaT,NaT,0 days 00:00:30.854000,0 days 00:00:28.154000,0 days 00:00:24.180000,0 days 01:13:09.290000,0 days 01:13:37.444000,0 days 01:14:01.624000,250.0,285.0,273.0,300.0,True,SOFT,24.0,False,Alpine,0 days 01:12:38.384000,2025-11-29 14:30:12.209,1,18.0,False,,False,True,2025,Qatar Grand Prix,83.188,86.4920,0 days 00:01:26.492000,83.188,0 days 00:01:23.188000,86.412563,0 days 00:01:26.412562500
8217,0 days 01:14:09.998000,STR,18,0 days 00:01:23.809000,19.0,2.0,NaT,NaT,0 days 00:00:31.077000,0 days 00:00:28.505000,0 days 00:00:24.227000,0 days 01:13:17.286000,0 days 01:13:45.791000,0 days 01:14:10.018000,243.0,286.0,279.0,303.0,False,SOFT,16.0,False,Aston Martin,0 days 01:12:46.189000,2025-11-29 14:30:20.014,1,19.0,False,,False,True,2025,Qatar Grand Prix,83.809,83.8130,0 days 00:01:23.813000,83.585,0 days 00:01:23.585000,87.410000,0 days 00:01:27.410000
8218,0 days 01:14:12.842000,COL,43,0 days 00:01:23.765000,19.0,2.0,NaT,NaT,0 days 00:00:30.945000,0 days 00:00:28.537000,0 days 00:00:24.283000,0 days 01:13:20.084000,0 days 01:13:48.621000,0 days 01:14:12.904000,249.0,284.0,271.0,303.0,True,SOFT,8.0,False,Alpine,0 days 01:12:49.077000,2025-11-29 14:30:22.902,1,20.0,False,,False,True,2025,Qatar Grand Prix,83.765,83.7650,0 days 00:01:23.765000,83.765,0 days 00:01:23.765000,83.765000,0 days 00:01:23.765000


In [12]:
transformed_driver=transformed_laps_driver.groupby(["Year","GP","Driver","Team","Compound"])[['LapTime (s)',
       'Median (s)', 'Median LapTime', 'Fastest (s)', 'Fastest LapTime',
       'Average (s)', 'Average LapTime']].min()

In [13]:
transformed_driver

LapTime (s)  \
Year GP               Driver Team            Compound                
2025 Qatar Grand Prix ALB    Williams        HARD           85.509   
                      ALO    Aston Martin    HARD           85.415   
                      ANT    Mercedes        HARD           84.464   
                      BEA    Haas F1 Team    MEDIUM         85.593   
                      BOR    Kick Sauber     HARD           85.663   
                      COL    Alpine          MEDIUM         85.878   
                                             SOFT           83.765   
                      GAS    Alpine          SOFT           83.188   
                      HAD    Racing Bulls    HARD           85.404   
                      HAM    Ferrari         MEDIUM         85.782   
                      HUL    Kick Sauber     MEDIUM         85.678   
                      LAW    Racing Bulls    MEDIUM         85.620   
                      LEC    Ferrari         HARD           85.638   
                      NOR    McLaren         HARD           84.507   
                      OCO    Haas F1 Team    MEDIUM         85.555   
                      PIA    McLaren         MEDIUM         83.988   
                      RUS    Mercedes        MEDIUM         84.564   
                      SAI    Williams        MEDIUM         85.463   
                      STR    Aston Martin    MEDIUM         85.753   
                                             SOFT           83.585   
                      TSU    Red Bull Racing HARD           84.816   
                      VER    Red Bull Racing HARD           84.541   

                                                       Median (s)  \
Year GP               Driver Team            Compound               
2025 Qatar Grand Prix ALB    Williams        HARD         86.0005   
                      ALO    Aston Martin    HARD         85.7610   
                      ANT    Mercedes        HARD         85.5315   
                      BEA    Haas F1 Team    MEDIUM       86.2110   
                      BOR    Kick Sauber     HARD         86.1385   
                      COL    Alpine          MEDIUM       86.6050   
                                             SOFT         83.7650   
                      GAS    Alpine          SOFT         86.4920   
                      HAD    Racing Bulls    HARD         85.9960   
                      HAM    Ferrari         MEDIUM       86.5675   
                      HUL    Kick Sauber     MEDIUM       86.3615   
                      LAW    Racing Bulls    MEDIUM       86.2040   
                      LEC    Ferrari         HARD         86.0050   
                      NOR    McLaren         HARD         84.9825   
                      OCO    Haas F1 Team    MEDIUM       86.2210   
                      PIA    McLaren         MEDIUM       84.8955   
                      RUS    Mercedes        MEDIUM       84.9450   
                      SAI    Williams        MEDIUM       85.9965   
                      STR    Aston Martin    MEDIUM       86.4340   
                                             SOFT         83.8130   
                      TSU    Red Bull Racing HARD         85.2755   
                      VER    Red Bull Racing HARD         85.0685   

                                                              Median LapTime  \
Year GP               Driver Team            Compound                          
2025 Qatar Grand Prix ALB    Williams        HARD     0 days 00:01:26.000500   
                      ALO    Aston Martin    HARD     0 days 00:01:25.761000   
                      ANT    Mercedes        HARD     0 days 00:01:25.531500   
                      BEA    Haas F1 Team    MEDIUM   0 days 00:01:26.211000   
                      BOR    Kick Sauber     HARD     0 days 00:01:26.138500   
                      COL    Alpine          MEDIUM   0 days 00:01:26.605000   
                                             SOFT     0 days 00:01:23.765000 